# 2.4 為什麼需要非線性？


如果一套配方只能把輸入乘權重再相加，多接一套配方是否就能做到更多事？不一定。假設第一套把x變成2x，第二套再乘3，合起來仍只是6x。即使中間用了多個特徵、加了偏移，兩層純Linear仍可合成一層Linear。這不是說多層程序沒執行，而是它能表示的規則沒有離開同一類加權配方。

有些規則需要輸入處於不同區域時做不同處理。比如我們要量「離零有多遠」，x=-2與x=2都應輸出2，x=0輸出0；任何單一的 `y=k*x+b` 都不能把這三點同時連上。因此需要一個會拐彎的運算，叫非線性（nonlinearity）。常用的ReLU會保留正數、把負數改成0：`ReLU(x)=max(0,x)`；它也叫激活函數（activation），表示夾在層與層之間的數值轉換，不是模型有意識地被激活。

下面用 [2.3](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html) 的Linear建立兩條中間線索x與-x，再看「直接相加」和「負的先清零再相加」會差多少。輸入三筆，分別為-2、0、2，每筆一特徵。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

a = nn.Linear(1, 2, bias=False)
b = nn.Linear(2, 1, bias=False)
with torch.no_grad():
    a.weight.copy_(torch.tensor([[1.0], [-1.0]]))
    b.weight.copy_(torch.tensor([[1.0, 1.0]]))
x = torch.tensor([[-2.0], [0.0], [2.0]])
hidden = a(x)
linear = b(hidden)
curved = b(F.relu(hidden))
combined = x @ (b.weight @ a.weight).T
print(hidden)
print("純線性", linear.squeeze(-1))
print("加ReLU", curved.squeeze(-1))
assert torch.allclose(linear, combined)

`bias=False` 省去偏移，讓例子只看權重。`hidden` 依序是 `[-2,2]`、`[0,0]`、`[2,-2]`。直接交給b，每列兩數相加，三個輸出全為0；合成權重 `b.weight @ a.weight` 也是0，最後檢查說明兩層確實能合成一層。`squeeze(-1)` 只去掉長度1的最後軸，方便印成一排三個結果。

加入 `F.relu` 後，三列變成 `[0,2]`、`[0,0]`、`[2,0]`，相加得到 `[2,0,2]`，正好表現距離零的大小。斜率描述輸入增加一單位時，輸出改變多少：左段輸入從-2增加到0，輸出從2降到0，每增加1就減少1；右段輸入從0增加到2，輸出從0升到2，每增加1就增加1。左右兩段的斜率分別為-1與1，一個固定加權的Linear無法同時表示這兩種變化。模型通常使用更多隱藏特徵與其他平滑轉換，但這個三點例子已把「多層配方」與「增加能拐彎的規則」分清。

非線性增加可表達的規則，不保證訓練一定找到好規則，也不保證新資料表現更好。參數更多、規則更多時，仍要按 [1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html) 留驗證資料。這裡沒有訓練，只是親手設定兩套權重說明什麼計算是可能的。

練習只把最後輸入2改成3，先預測純線性仍0、ReLU輸出3，再執行核對。接着把 `F.relu(hidden)` 改成 `hidden`，確認它退回純線性結果。這兩次操作讓你定位真正改變行為的是哪一行，而不是看到層數增加就猜能力也必然增加。
